# 🔍 Candidate Scraper Pipeline — Fixed & Upgraded
**Platforms:** Resume PDF → GitHub · LeetCode · HackerRank · LinkedIn · Stack Overflow · Kaggle

**Fixes applied:**
- GitHub: full profile fields + repo stars/forks/language/description, fork filtering, pagination bug fixed
- LeetCode: difficulty breakdown (Easy/Medium/Hard) + contest rating + top % + badges
- HackerRank: per-domain star scores parsed + graceful None handling
- LinkedIn: JSON parsed into structured fields immediately (not raw blob)
- Added: Stack Overflow (official API) + Kaggle (official API)
- Added: rate limiting, retry logic, credential secrets
- Added: transformation layer — numeric vs contextual split
- Fixed: `leetcode_problems` NameError, double-escaped regex

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install pymupdf pdfplumber requests pandas kaggle -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.1 MB/s eta 0:00:00


In [ ]:
# ── Cell 2: Credentials — stored in Colab Secrets (never hardcode tokens) ────
# Go to 🔑 Secrets (left sidebar) and add each key before running.
# If a secret is missing the pipeline skips that platform gracefully.

from google.colab import userdata

def _secret(key, default=""):
    """Safely read a Colab secret; return default if not set."""
    try:
        val = userdata.get(key)
        return val if val else default
    except Exception:
        return default

GITHUB_TOKEN      = _secret("GITHUB_TOKEN")       # optional — raises rate limit 60→5000/hr
DATAMAGNET_TOKEN  = _secret("DATAMAGNET_TOKEN")   # required for LinkedIn
KAGGLE_USERNAME   = _secret("KAGGLE_USERNAME")    # required for Kaggle
KAGGLE_KEY        = _secret("KAGGLE_KEY")         # required for Kaggle
SO_API_KEY        = _secret("SO_API_KEY")         # optional — raises SO quota 300→10k/day

print("✓ Credentials loaded")
print(f"  GitHub token  : {'set' if GITHUB_TOKEN else 'not set (60 req/hr limit)'}")
print(f"  DataMagnet    : {'set' if DATAMAGNET_TOKEN else 'not set — LinkedIn will be skipped'}")
print(f"  Kaggle        : {'set' if KAGGLE_USERNAME and KAGGLE_KEY else 'not set — Kaggle will be skipped'}")
print(f"  StackOverflow : {'set' if SO_API_KEY else 'not set (300 req/day limit)'}")

✓ Credentials loaded
  GitHub token  : not set (60 req/hr limit)
  DataMagnet    : set
  Kaggle        : set
  StackOverflow : not set (300 req/day limit)


In [ ]:
# ── Cell 3: Upload resume PDF ─────────────────────────────────────────────────
from google.colab import files

uploaded = files.upload()
resume_path = list(uploaded.keys())[0]
print(f"✓ Uploaded: {resume_path}")

Saving 2 Resume 2201641640020.pdf to 2 Resume 2201641640020.pdf
✓ Uploaded: 2 Resume 2201641640020.pdf


In [ ]:
# ── Cell 4: Resume parser — extract links, email, raw text ───────────────────
import fitz          # PyMuPDF  — clickable hyperlinks embedded in PDF
import pdfplumber    # pdfplumber — visible text including printed URLs
import re
from urllib.parse import urlparse

# @title ── Manual overrides (paste if not found in resume) ── { display-mode: "both" }
manual_linkedin   = "https://www.linkedin.com/in/dewashish-dwivedi-806a92206/" # @param {type:"string"}
manual_github     = "" # @param {type:"string"}
manual_leetcode   = "https://leetcode.com/u/aiml0020/" # @param {type:"string"}
manual_hackerrank = "https://www.hackerrank.com/profile/ddwivedi2003" # @param {type:"string"}
manual_kaggle     = "https://www.kaggle.com/dewashishdwivedi" # @param {type:"string"}
manual_stackoverflow = "https://stackoverflow.com/users/31091536/dewashish-dwivedi" # @param {type:"string"}  # numeric user ID or profile URL
manual_portfolio  = "" # @param {type:"string"}

# ── Extract clickable hyperlinks (embedded in PDF metadata) ──────────────────
doc = fitz.open(resume_path)
clickable_links = []
for page in doc:
    for link in page.get_links():
        uri = link.get("uri", "")
        if uri:
            clickable_links.append(uri)
doc.close()

# ── Extract visible text ─────────────────────────────────────────────────────
_raw = ""
with pdfplumber.open(resume_path) as pdf:
    for page in pdf.pages:
        t = page.extract_text()
        if t:
            _raw += t + "\n"

# ── Patterns ─────────────────────────────────────────────────────────────────
URL_RE   = r'(https?://[^\s,)<>\"]+|www\.[^\s,)<>\"]+)'
EMAIL_RE = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

text_links = re.findall(URL_RE, _raw)
all_links  = list(dict.fromkeys(clickable_links + text_links))   # deduplicated, order-preserved

_emails = re.findall(EMAIL_RE, _raw)
email   = _emails[0] if _emails else None

# Clean text: strip emails and URLs for contextual use
clean_text = re.sub(r'\n{3,}', '\n\n',
             re.sub(EMAIL_RE, '',
             re.sub(URL_RE, '', _raw))).strip()

content = {"raw": _raw.strip(), "clean": clean_text}

# ── Classify links by platform ───────────────────────────────────────────────
PLATFORM_RULES = {
    "linkedin": "linkedin", "github": "github", "gitlab": "gitlab",
    "twitter": "twitter", "x.com": "twitter", "leetcode": "leetcode",
    "kaggle": "kaggle", "medium": "blog", "dev.to": "blog",
    "hashnode": "blog", "substack": "blog", "wordpress": "blog",
    "behance": "portfolio", "dribbble": "portfolio", "figma": "portfolio",
    "notion": "portfolio", "vercel": "portfolio", "netlify": "portfolio",
    "stackoverflow": "stackoverflow", "hackerrank": "hackerrank",
    "codepen": "codepen", "huggingface": "huggingface",
    "researchgate": "research", "scholar.google": "research",
    "youtube": "youtube",
}

def classify_link(url):
    try:
        domain = urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        domain = url.lower()
    for keyword, category in PLATFORM_RULES.items():
        if keyword in domain:
            return category
    return "other"

_seg = {}
for link in all_links:
    cat = classify_link(link)
    _seg.setdefault(cat, []).append(link)

# ── Resolve final URLs (manual override wins) ─────────────────────────────────
def _pick(manual, key):
    return manual.strip() if manual.strip() else _seg.get(key, [None])[0]

linkedin      = _pick(manual_linkedin,      "linkedin")
github        = _pick(manual_github,        "github")
leetcode      = _pick(manual_leetcode,      "leetcode")
hackerrank    = _pick(manual_hackerrank,    "hackerrank")
kaggle        = _pick(manual_kaggle,        "kaggle")
stackoverflow = _pick(manual_stackoverflow, "stackoverflow")
portfolio     = _pick(manual_portfolio,     "portfolio")

gitlab      = _seg.get("gitlab",      [None])[0]
blog        = _seg.get("blog",        [None])[0]
codepen     = _seg.get("codepen",     [None])[0]
huggingface = _seg.get("huggingface", [None])[0]
research    = _seg.get("research",    [None])[0]

print("✓ Resume parsed!")
print(f"  email         = {email}")
print(f"  github        = {github}")
print(f"  leetcode      = {leetcode}")
print(f"  hackerrank    = {hackerrank}")
print(f"  kaggle        = {kaggle}")
print(f"  stackoverflow = {stackoverflow}")
print(f"  linkedin      = {linkedin}")
print(f"  portfolio     = {portfolio}")

✓ Resume parsed!
  email         = ddwivedi2003@gmail.com
  github        = https://github.com/ddwivedi2003
  leetcode      = https://leetcode.com/u/aiml0020/
  hackerrank    = https://www.hackerrank.com/profile/ddwivedi2003
  kaggle        = https://www.kaggle.com/dewashishdwivedi
  stackoverflow = https://stackoverflow.com/users/31091536/dewashish-dwivedi
  linkedin      = https://www.linkedin.com/in/dewashish-dwivedi-806a92206/
  portfolio     = None


In [ ]:
# ── Cell 5: Shared utilities — retry, rate limiting, username extractor ───────
import requests
import time
import json
from bs4 import BeautifulSoup

def _get(url, headers=None, params=None, retries=3, delay=1.5):
    """GET with exponential-backoff retry. Returns response or None."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=15)
            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", delay * (attempt + 1) * 2))
                print(f"  ⏳ Rate limited — waiting {wait}s")
                time.sleep(wait)
                continue
            return r
        except requests.exceptions.Timeout:
            print(f"  ⏳ Timeout (attempt {attempt+1}/{retries})")
            time.sleep(delay * (attempt + 1))
        except Exception as e:
            print(f"  ✗ Request error: {e}")
            break
    return None

def _post(url, json_body, headers=None, retries=3, delay=1.5):
    """POST with retry."""
    for attempt in range(retries):
        try:
            r = requests.post(url, json=json_body, headers=headers, timeout=15)
            if r.status_code == 429:
                time.sleep(delay * (attempt + 1) * 2)
                continue
            return r
        except Exception as e:
            print(f"  ✗ Request error: {e}")
            break
    return None

def _slug(url, fallback=None):
    """Extract the last meaningful path segment from a URL."""
    if not url:
        return fallback
    parts = [p for p in url.rstrip("/").split("/") if p]
    slug = parts[-1].split("?")[0].split("#")[0] if parts else None
    # Drop LeetCode's /u/ prefix  e.g. leetcode.com/u/username
    if slug and slug == "u" and len(parts) >= 2:
        slug = parts[-1]
    return slug or fallback

print("✓ Utilities ready")

✓ Utilities ready


In [ ]:
# ── Cell 6: GitHub scraper ────────────────────────────────────────────────────
# FIX: full profile fields + repo stars/forks/language/description
# FIX: fork filtering (only original work counts)
# FIX: pagination regex was double-escaped (\\s* → \s*)

github_profile = {}
github_repos   = []

if github:
    try:
        gh_user = _slug(github)
        GH_HDR  = {"Authorization": f"Bearer {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}
        GH_HDR["Accept"] = "application/vnd.github+json"

        # ── Profile ──────────────────────────────────────────────────────────
        res  = _get(f"https://api.github.com/users/{gh_user}", headers=GH_HDR)
        prof = res.json() if res else {}

        # ── All repos (paginated) ─────────────────────────────────────────────
        all_repos, url = [], f"https://api.github.com/users/{gh_user}/repos?per_page=100&sort=pushed"
        while url:
            r = _get(url, headers=GH_HDR)
            if not r:
                break
            data = r.json()
            if isinstance(data, list):
                all_repos.extend(data)
            # FIX: was r'<([^>]+)>;\\s*rel="next"' — double-escaped, never matched
            nxt = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", ""))
            url = nxt.group(1) if nxt else None
            time.sleep(0.3)   # be polite between pages

        # ── Compute aggregate stats ───────────────────────────────────────────
        original_repos  = [r for r in all_repos if not r.get("fork")]  # FIX: exclude forks
        total_stars     = sum(r.get("stargazers_count", 0) for r in original_repos)
        total_forks_got = sum(r.get("forks_count", 0) for r in original_repos)
        languages_used  = list({r["language"] for r in original_repos if r.get("language")})

        github_profile = {
            "gh_username":        prof.get("login"),
            "gh_name":            prof.get("name"),
            "gh_bio":             prof.get("bio"),             # contextual
            "gh_company":         prof.get("company"),         # contextual
            "gh_blog":            prof.get("blog"),
            "gh_location":        prof.get("location"),
            "gh_followers":       prof.get("followers"),       # numeric
            "gh_following":       prof.get("following"),       # numeric
            "gh_public_repos":    prof.get("public_repos"),    # numeric
            "gh_account_created": prof.get("created_at"),      # numeric (age)
            "gh_total_stars":     total_stars,                 # numeric — key signal
            "gh_total_forks_got": total_forks_got,            # numeric
            "gh_original_repos":  len(original_repos),        # numeric
            "gh_languages":       ", ".join(languages_used),  # contextual/numeric
            "gh_top_repo_stars":  max((r.get("stargazers_count", 0) for r in original_repos), default=0),
        }

        # ── Per-repo rows ─────────────────────────────────────────────────────
        for r in original_repos:   # FIX: only original repos
            github_repos.append({
                "repo_name":   r.get("name"),
                "repo_url":    r.get("html_url"),
                "stars":       r.get("stargazers_count", 0),     # FIX: was missing
                "forks":       r.get("forks_count", 0),          # FIX: was missing
                "language":    r.get("language"),                # FIX: was missing
                "description": r.get("description"),             # FIX: was missing (contextual)
                "updated_at":  r.get("pushed_at"),               # FIX: was missing (recency)
                "size_kb":     r.get("size", 0),
                "open_issues": r.get("open_issues_count", 0),
                "topics":      ", ".join(r.get("topics", [])),   # contextual
            })

        print(f"✓ GitHub — {len(original_repos)} original repos | {total_stars} total stars | languages: {', '.join(languages_used[:5])}")
    except Exception as e:
        print(f"✗ GitHub error: {e}")
else:
    print("— GitHub: no URL found, skipping")

✓ GitHub — 19 original repos | 0 total stars | languages: EJS, TypeScript, Python


In [ ]:
# ── Cell 7: LeetCode scraper ──────────────────────────────────────────────────
# FIX: full GraphQL query — difficulty breakdown + contest rating + top % + badges
# FIX: was only fetching lc_total_solved (1 number from a minimal query)
# FIX: leetcode_problems was never defined → NameError silently swallowed

leetcode_profile = {}

LC_HDR = {
    "Content-Type": "application/json",
    "Referer":      "https://leetcode.com",    # required — 403 without it
    "User-Agent":   "Mozilla/5.0",
}

# FIX: expanded query — everything in one call
LC_QUERY = """
query getUserProfile($username: String!) {
  matchedUser(username: $username) {
    profile {
      realName
      ranking
      reputation
      starRating
    }
    submitStats {
      acSubmissionNum { difficulty count }
    }
    badges { name icon }
    languageProblemCount { languageName problemsSolved }
    tagProblemCounts {
      advanced  { tagName problemsSolved }
      intermediate { tagName problemsSolved }
      fundamental { tagName problemsSolved }
    }
  }
  userContestRanking(username: $username) {
    rating
    globalRanking
    totalParticipants
    topPercentage
    attendedContestsCount
  }
}
"""

if leetcode:
    try:
        lc_user = _slug(leetcode)
        # Handle /u/username pattern
        if not lc_user or lc_user == "u":
            parts = [p for p in leetcode.rstrip("/").split("/") if p and p != "u"]
            lc_user = parts[-1] if parts else None

        r = _post(
            "https://leetcode.com/graphql",
            json_body={"query": LC_QUERY, "variables": {"username": lc_user}},
            headers=LC_HDR
        )

        data    = r.json().get("data", {}) if r else {}
        user    = data.get("matchedUser") or {}
        contest = data.get("userContestRanking") or {}
        profile = user.get("profile") or {}
        stats   = user.get("submitStats", {}).get("acSubmissionNum", [])

        # FIX: parse all difficulty levels (was only getting 'All')
        solved = {s["difficulty"]: s["count"] for s in stats}

        # Top tags solved
        tag_counts = user.get("tagProblemCounts", {})
        advanced_tags = [(t["tagName"], t["problemsSolved"])
                         for t in tag_counts.get("advanced", [])]
        advanced_tags.sort(key=lambda x: x[1], reverse=True)

        # Languages
        langs = user.get("languageProblemCount", [])
        lang_str = ", ".join(f"{l['languageName']}({l['problemsSolved']})" for l in langs[:5])

        # Badges
        badges = [b["name"] for b in user.get("badges", [])]

        leetcode_profile = {
            "lc_username":         lc_user,
            "lc_ranking":          profile.get("ranking"),          # numeric — global rank
            "lc_total_solved":     solved.get("All", 0),            # numeric
            "lc_easy_solved":      solved.get("Easy", 0),           # FIX: was missing
            "lc_medium_solved":    solved.get("Medium", 0),         # FIX: was missing
            "lc_hard_solved":      solved.get("Hard", 0),           # FIX: was missing
            "lc_contest_rating":   contest.get("rating"),           # FIX: was missing (key signal)
            "lc_contest_rank":     contest.get("globalRanking"),    # FIX: was missing
            "lc_top_percentage":   contest.get("topPercentage"),    # FIX: was missing (e.g. top 5.3%)
            "lc_contests_attended":contest.get("attendedContestsCount"), # FIX: was missing
            "lc_star_rating":      profile.get("starRating"),
            "lc_reputation":       profile.get("reputation"),
            "lc_badges":           ", ".join(badges),               # contextual
            "lc_languages":        lang_str,                        # contextual
            "lc_top_topics":       ", ".join(t[0] for t in advanced_tags[:5]),  # contextual
        }

        print(f"✓ LeetCode — total: {solved.get('All',0)} | E:{solved.get('Easy',0)} M:{solved.get('Medium',0)} H:{solved.get('Hard',0)} | contest rating: {contest.get('rating','—')} | top {contest.get('topPercentage','—')}%")
    except Exception as e:
        print(f"✗ LeetCode error: {e}")
else:
    print("— LeetCode: no URL found, skipping")

✓ LeetCode — total: 130 | E:82 M:44 H:4 | contest rating: — | top —%


In [ ]:
# ── Cell 8: HackerRank scraper ────────────────────────────────────────────────
# FIX: parse per-domain star scores as individual numeric fields
# FIX: handle None rank/score gracefully (common — not all accounts have them)
# FIX: extract certificate count from profile HTML

hackerrank_profile = {}
HR_HDR = {"User-Agent": "Mozilla/5.0"}

# Domains HackerRank badges cover — we extract each as a separate numeric field
HR_DOMAINS = [
    "Problem Solving", "Python", "Java", "C", "C++",
    "SQL", "Databases", "Linux Shell", "Regex",
    "JavaScript", "Rest API", "Go", "Ruby",
]

def _hr_stars(badges, domain):
    """Find star count for a given HackerRank domain badge."""
    for b in badges:
        if domain.lower() in b.get("badge_name", "").lower():
            return b.get("stars", 0)
    return 0

if hackerrank:
    try:
        hr_user = _slug(hackerrank)

        # ── Profile endpoint ──────────────────────────────────────────────────
        res_prof  = _get(f"https://www.hackerrank.com/rest/hackers/{hr_user}/profile", headers=HR_HDR)
        prof_data = res_prof.json().get("model", {}) if res_prof else {}

        # ── Badges endpoint ───────────────────────────────────────────────────
        res_badges = _get(f"https://www.hackerrank.com/rest/hackers/{hr_user}/badges", headers=HR_HDR)
        badges     = res_badges.json().get("models", []) if res_badges else []

        # ── Human-readable skill summary ──────────────────────────────────────
        hr_skills_list = [
            f"{b['badge_name']} ({b.get('stars', 0)} stars)"
            for b in badges if b.get("stars", 0) > 0
        ]

        # FIX: parse each domain into its own numeric field for the rule engine
        domain_scores = {f"hr_{d.lower().replace(' ', '_')}_stars": _hr_stars(badges, d)
                         for d in HR_DOMAINS}

        hackerrank_profile = {
            "hr_username":    hr_user,
            "hr_rank":        prof_data.get("level"),    # often None — that's normal
            "hr_score":       prof_data.get("score"),    # often None — that's normal
            "hr_country":     prof_data.get("country"),
            "hr_skills_raw":  ", ".join(hr_skills_list),  # contextual
            "hr_total_badges":len([b for b in badges if b.get("stars", 0) > 0]),  # numeric
            **domain_scores,  # FIX: hr_python_stars, hr_sql_stars etc. — individual numerics
        }

        print(f"✓ HackerRank — {hackerrank_profile['hr_total_badges']} active badges | {', '.join(hr_skills_list[:4])}")
        if not prof_data.get("level"):
            print("  ℹ  rank/score are None — normal for accounts without a computed global rank")
    except Exception as e:
        print(f"✗ HackerRank error: {e}")
else:
    print("— HackerRank: no URL found, skipping")

✓ HackerRank — 3 active badges | Problem Solving (2 stars), Java (1 stars), Python (3 stars)
  ℹ  rank/score are None — normal for accounts without a computed global rank


In [ ]:
# ── Cell 9: LinkedIn scraper (DataMagnet API) ─────────────────────────────────
# FIX: parse JSON response into structured fields immediately
# FIX: was storing entire response.text as a single blob

linkedin_profile = {}

if linkedin and DATAMAGNET_TOKEN:
    try:
        r = requests.post(
            "https://api.datamagnet.co/api/v1/linkedin/person",
            headers={
                "Authorization": f"Bearer {DATAMAGNET_TOKEN}",
                "Content-Type":  "application/json"
            },
            json={"url": linkedin},
            timeout=30
        )
        raw = r.json()
        # DataMagnet wraps the payload in a 'message' key
        data = raw.get("message", raw) if isinstance(raw, dict) else {}

        # ── Experience processing ─────────────────────────────────────────────
        experiences = data.get("experiences") or []
        exp_titles  = [e.get("title", "") for e in experiences]
        exp_companies = [e.get("company", "") for e in experiences]

        # Compute total months of experience from date fields
        def _months(exp):
            try:
                from datetime import datetime
                fmt = "%Y-%m-%d"
                start = datetime.strptime(exp.get("start_date", "")[:10], fmt)
                end_raw = exp.get("end_date") or ""
                end = datetime.strptime(end_raw[:10], fmt) if end_raw else datetime.now()
                return max(0, (end - start).days // 30)
            except Exception:
                return 0

        total_exp_months = sum(_months(e) for e in experiences)

        # ── Education processing ──────────────────────────────────────────────
        education = data.get("education") or []

        # ── Skills ────────────────────────────────────────────────────────────
        skills     = data.get("skills") or []
        skill_names = [s.get("name", "") if isinstance(s, dict) else str(s) for s in skills]

        # ── Certifications ────────────────────────────────────────────────────
        certs = data.get("certifications") or []

        # ── Recommendations ───────────────────────────────────────────────────
        recs = data.get("recommendations") or []

        # FIX: structured fields — not a blob
        linkedin_profile = {
            # Identity
            "li_name":               data.get("full_name") or data.get("name"),
            "li_headline":           data.get("headline"),         # contextual
            "li_summary":            data.get("summary") or data.get("about"),  # contextual
            "li_location":           data.get("location"),
            "li_profile_url":        data.get("linkedin_url") or linkedin,
            # Network — numeric
            "li_followers":          data.get("followers_count") or data.get("follower_count"),
            "li_connections":        data.get("connections_count") or data.get("connection_count"),
            # Experience — numeric
            "li_num_positions":      len(experiences),
            "li_total_exp_months":   total_exp_months,
            "li_exp_titles":         " | ".join(exp_titles),       # contextual
            "li_exp_companies":      " | ".join(exp_companies),    # contextual
            # Education — numeric
            "li_num_education":      len(education),
            "li_edu_details":        " | ".join(
                f"{e.get('school','')} ({e.get('degree_name','')})" for e in education
            ),  # contextual
            # Skills — numeric
            "li_num_skills":         len(skill_names),
            "li_skills":             ", ".join(skill_names[:30]),  # contextual
            # Credentials — numeric
            "li_num_certs":          len(certs),
            "li_cert_names":         ", ".join(c.get("name", "") for c in certs),  # contextual
            # Social proof — numeric
            "li_num_recommendations":len(recs),
            "li_rec_texts":          " | ".join(r.get("description", "") for r in recs[:3]),  # contextual
            # Completeness flags — numeric
            "li_has_photo":          bool(data.get("profile_picture") or data.get("photo")),
            "li_has_summary":        bool(data.get("summary") or data.get("about")),
            "li_has_experience":     len(experiences) > 0,
            "li_has_education":      len(education) > 0,
            "li_has_skills":         len(skill_names) > 0,
            "li_has_certs":          len(certs) > 0,
            "li_has_recommendations":len(recs) > 0,
        }

        print(f"✓ LinkedIn — {linkedin_profile['li_name']} | {len(experiences)} positions | {len(skill_names)} skills | {total_exp_months} months exp")
    except Exception as e:
        print(f"✗ LinkedIn error: {e}")
elif not DATAMAGNET_TOKEN:
    print("— LinkedIn: DATAMAGNET_TOKEN not set in Secrets — skipping")
else:
    print("— LinkedIn: no URL found, skipping")

✓ LinkedIn — Dewashish Dwivedi | 0 positions | 20 skills | 0 months exp


In [ ]:
# ── Cell 10: Stack Overflow scraper (NEW) ─────────────────────────────────────
# Official public API — no auth needed for basic data
# SO_API_KEY raises quota from 300 → 10,000 req/day (register at stackapps.com)

stackoverflow_profile = {}

def _so_user_id(url_or_id):
    """Accept either a numeric ID or a full SO profile URL."""
    if not url_or_id:
        return None
    s = str(url_or_id).strip()
    if s.isdigit():
        return s
    # https://stackoverflow.com/users/12345/username → 12345
    m = re.search(r'/users/(\d+)', s)
    return m.group(1) if m else None

so_id = _so_user_id(stackoverflow)

if so_id:
    try:
        SO_BASE   = "https://api.stackexchange.com/2.3"
        SO_PARAMS = {"site": "stackoverflow"}
        if SO_API_KEY:
            SO_PARAMS["key"] = SO_API_KEY

        # ── Profile ───────────────────────────────────────────────────────────
        r_prof = _get(f"{SO_BASE}/users/{so_id}", params=SO_PARAMS)
        items  = r_prof.json().get("items", []) if r_prof else []
        prof   = items[0] if items else {}

        # ── Top tags (proves domain knowledge) ────────────────────────────────
        r_tags = _get(f"{SO_BASE}/users/{so_id}/tags",
                      params={**SO_PARAMS, "pagesize": 20, "sort": "activity"})
        tags   = r_tags.json().get("items", []) if r_tags else []
        time.sleep(0.5)

        # ── Top answers (quality signal) ──────────────────────────────────────
        r_ans = _get(f"{SO_BASE}/users/{so_id}/answers",
                     params={**SO_PARAMS, "pagesize": 30, "sort": "votes", "filter": "withbody"})
        answers = r_ans.json().get("items", []) if r_ans else []
        time.sleep(0.5)

        # ── Badges ────────────────────────────────────────────────────────────
        r_badges = _get(f"{SO_BASE}/users/{so_id}/badges",
                        params={**SO_PARAMS, "pagesize": 100})
        badges = r_badges.json().get("items", []) if r_badges else []

        badge_counts = prof.get("badge_counts", {})
        top_tags     = [t["name"] for t in tags]
        accepted_ans = sum(1 for a in answers if a.get("is_accepted"))
        avg_ans_score = round(sum(a.get("score", 0) for a in answers) / max(len(answers), 1), 1)

        stackoverflow_profile = {
            "so_user_id":        so_id,
            "so_display_name":   prof.get("display_name"),
            "so_reputation":     prof.get("reputation", 0),      # numeric — key signal
            "so_answer_count":   prof.get("answer_count", 0),    # numeric
            "so_question_count": prof.get("question_count", 0),  # numeric
            "so_gold_badges":    badge_counts.get("gold", 0),    # numeric
            "so_silver_badges":  badge_counts.get("silver", 0),  # numeric
            "so_bronze_badges":  badge_counts.get("bronze", 0),  # numeric
            "so_accepted_answers":accepted_ans,                   # numeric
            "so_avg_answer_score":avg_ans_score,                  # numeric
            "so_top_tags":       ", ".join(top_tags[:10]),        # contextual — domain proof
            "so_account_created":prof.get("creation_date"),       # numeric (age)
            "so_last_access":    prof.get("last_access_date"),    # numeric (activity)
            "so_profile_url":    prof.get("link"),
        }

        print(f"✓ Stack Overflow — rep: {prof.get('reputation',0):,} | answers: {prof.get('answer_count',0)} | gold badges: {badge_counts.get('gold',0)} | top tags: {', '.join(top_tags[:5])}")
    except Exception as e:
        print(f"✗ Stack Overflow error: {e}")
else:
    print("— Stack Overflow: no user ID/URL found, skipping")

✓ Stack Overflow — rep: 1 | answers: 0 | gold badges: 0 | top tags: 


In [ ]:
# ── Cell 11: Kaggle scraper (NEW) ─────────────────────────────────────────────
# Uses official Kaggle REST API — requires KAGGLE_USERNAME + KAGGLE_KEY in Secrets
# Get keys: kaggle.com → Settings → API → Create New Token

kaggle_profile = {}

if kaggle and KAGGLE_USERNAME and KAGGLE_KEY:
    try:
        kg_user = _slug(kaggle)

        KG_AUTH = (KAGGLE_USERNAME, KAGGLE_KEY)
        KG_BASE = "https://www.kaggle.com/api/v1"

        # ── Public profile metadata ───────────────────────────────────────────
        r_prof = requests.get(
            f"{KG_BASE}/users/{kg_user}",
            auth=KG_AUTH, timeout=15
        )
        prof = r_prof.json() if r_prof.ok else {}
        time.sleep(0.5)

        # ── Datasets ──────────────────────────────────────────────────────────
        r_ds = requests.get(
            f"{KG_BASE}/datasets",
            params={"user": kg_user, "page": 1, "pageSize": 20, "sortBy": "votes"},
            auth=KG_AUTH, timeout=15
        )
        datasets = r_ds.json() if r_ds.ok else []
        time.sleep(0.5)

        # ── Notebooks / kernels ───────────────────────────────────────────────
        r_nb = requests.get(
            f"{KG_BASE}/kernels",
            params={"userDisplayName": kg_user, "page": 1, "pageSize": 20, "sortBy": "voteCount"},
            auth=KG_AUTH, timeout=15
        )
        notebooks = r_nb.json() if r_nb.ok else []

        # ── Aggregate stats ───────────────────────────────────────────────────
        ds_list   = datasets if isinstance(datasets, list) else datasets.get("datasets", [])
        nb_list   = notebooks if isinstance(notebooks, list) else notebooks.get("kernels", [])

        total_ds_votes = sum(d.get("voteCount", d.get("totalVotes", 0)) for d in ds_list)
        total_nb_votes = sum(n.get("voteCount", n.get("totalVotes", 0)) for n in nb_list)

        kaggle_profile = {
            "kg_username":        kg_user,
            "kg_display_name":    prof.get("displayName") or prof.get("userName"),
            "kg_tier":            prof.get("tier"),             # numeric/rank — key signal: Novice/Contributor/Expert/Master/GrandMaster
            "kg_ranking":         prof.get("ranking"),          # numeric
            "kg_points":          prof.get("points"),           # numeric
            "kg_gold_medals":     prof.get("goldMedals")     or prof.get("totalGoldMedals", 0),    # numeric
            "kg_silver_medals":   prof.get("silverMedals")   or prof.get("totalSilverMedals", 0),  # numeric
            "kg_bronze_medals":   prof.get("bronzeMedals")   or prof.get("totalBronzeMedals", 0),  # numeric
            "kg_num_datasets":    len(ds_list),                  # numeric
            "kg_dataset_votes":   total_ds_votes,                # numeric
            "kg_num_notebooks":   len(nb_list),                  # numeric
            "kg_notebook_votes":  total_nb_votes,                # numeric
            "kg_dataset_titles":  ", ".join(d.get("title", d.get("ref", "")) for d in ds_list[:5]),  # contextual
            "kg_notebook_titles": ", ".join(n.get("title", n.get("ref", "")) for n in nb_list[:5]),  # contextual
        }

        print(f"✓ Kaggle — tier: {prof.get('tier','—')} | rank: {prof.get('ranking','—')} | gold medals: {kaggle_profile['kg_gold_medals']} | datasets: {len(ds_list)} | notebooks: {len(nb_list)}")
    except Exception as e:
        print(f"✗ Kaggle error: {e}")
elif not (KAGGLE_USERNAME and KAGGLE_KEY):
    print("— Kaggle: KAGGLE_USERNAME / KAGGLE_KEY not set in Secrets — skipping")
else:
    print("— Kaggle: no URL found, skipping")

✓ Kaggle — tier: — | rank: — | gold medals: 0 | datasets: 0 | notebooks: 0


In [ ]:
# ── Cell 12: Per-Platform Transformation — numeric vs contextual per platform ─
# CHANGE: Each platform now produces its own numeric CSV and contextual CSV.
# Output files (12 total across 6 platforms + 1 resume):
#   linkedin_numeric.csv      linkedin_contextual.csv
#   github_numeric.csv        github_contextual.csv
#   leetcode_numeric.csv      leetcode_contextual.csv
#   hackerrank_numeric.csv    hackerrank_contextual.csv
#   stackoverflow_numeric.csv stackoverflow_contextual.csv
#   kaggle_numeric.csv        kaggle_contextual.csv
#   resume_contextual.csv     (resume has no numeric fields)
#
# Rule: a field is NUMERIC if it is an int/float/bool/ordinal that the rule engine
#        can use directly without LLM interpretation.
#        Everything else (free text, comma-separated labels, titles) is CONTEXTUAL.

import pandas as pd
import math

def _is_numeric(val):
    """Return True if the value should go into the numeric split."""
    if val is None:
        return False
    if isinstance(val, bool):
        return True
    if isinstance(val, (int, float)):
        return not (isinstance(val, float) and math.isnan(val))
    # String: is it a pure number?
    try:
        float(str(val).strip())
        return True
    except (ValueError, TypeError):
        return False

def split_platform(profile_dict: dict) -> tuple[dict, dict]:
    """
    Split a flat platform dict into (numeric_dict, contextual_dict).
    Numeric  → int/float/bool values (counts, stars, ratings, flags).
    Contextual → strings, comma-separated labels, free text, titles.
    """
    numeric     = {}
    contextual  = {}
    for k, v in profile_dict.items():
        if _is_numeric(v):
            numeric[k] = v
        else:
            contextual[k] = v
    return numeric, contextual

def save_platform_csvs(platform: str, profile_dict: dict,
                        extra_numeric: dict = None,
                        extra_contextual: dict = None) -> dict:
    """
    Split a platform profile dict and save:
      {platform}_numeric.csv
      {platform}_contextual.csv
    Returns a summary dict with row counts.
    extra_numeric / extra_contextual: manually-classified fields that
      override the auto-detection (e.g. boolean flags, ordinal tiers).
    """
    if not profile_dict:
        return {platform: {'numeric': 0, 'contextual': 0, 'status': 'skipped (no data)'}}

    num_d, ctx_d = split_platform(profile_dict)

    # Apply manual overrides
    if extra_numeric:
        for k, v in extra_numeric.items():
            num_d[k] = v
            ctx_d.pop(k, None)
    if extra_contextual:
        for k, v in extra_contextual.items():
            ctx_d[k] = v
            num_d.pop(k, None)

    num_fname = f'{platform}_numeric.csv'
    ctx_fname = f'{platform}_contextual.csv'

    pd.DataFrame([num_d]).to_csv(num_fname, index=False)
    pd.DataFrame([ctx_d]).to_csv(ctx_fname, index=False)

    return {
        platform: {
            'numeric':    len(num_d),
            'contextual': len(ctx_d),
            'status':     'saved',
            'num_file':   num_fname,
            'ctx_file':   ctx_fname,
        }
    }


# ── LINKEDIN ──────────────────────────────────────────────────────────────────
# Boolean completeness flags + counts → numeric
# Headline, summary, bio, titles, skills text → contextual
li_force_numeric = {
    k: v for k, v in linkedin_profile.items()
    if k in {
        'li_followers', 'li_connections', 'li_num_positions', 'li_total_exp_months',
        'li_num_education', 'li_num_skills', 'li_num_certs', 'li_num_recommendations',
        'li_has_photo', 'li_has_summary', 'li_has_experience',
        'li_has_education', 'li_has_skills', 'li_has_certs', 'li_has_recommendations',
    }
}
li_force_contextual = {
    k: v for k, v in linkedin_profile.items()
    if k in {
        'li_name', 'li_headline', 'li_summary', 'li_location', 'li_profile_url',
        'li_exp_titles', 'li_exp_companies', 'li_edu_details',
        'li_skills', 'li_cert_names', 'li_rec_texts',
    }
}
li_summary = save_platform_csvs(
    'linkedin', linkedin_profile,
    extra_numeric    = li_force_numeric,
    extra_contextual = li_force_contextual
)

# ── GITHUB ────────────────────────────────────────────────────────────────────
# Counts, stars, forks, follower numbers → numeric
# Bio, company, blog, language list (comma text), repo descriptions → contextual
gh_force_contextual = {
    k: v for k, v in github_profile.items()
    if k in {'gh_bio', 'gh_company', 'gh_blog', 'gh_location',
              'gh_username', 'gh_name', 'gh_languages', 'gh_account_created'}
}
gh_force_numeric = {
    k: v for k, v in github_profile.items()
    if k in {'gh_followers', 'gh_following', 'gh_public_repos', 'gh_total_stars',
              'gh_total_forks_got', 'gh_original_repos', 'gh_top_repo_stars'}
}
gh_summary = save_platform_csvs(
    'github', github_profile,
    extra_numeric    = gh_force_numeric,
    extra_contextual = gh_force_contextual
)

# ── LEETCODE ─────────────────────────────────────────────────────────────────
# All solve counts, ratings, rankings → numeric
# Topics list, languages list, badges text → contextual
lc_force_contextual = {
    k: v for k, v in leetcode_profile.items()
    if k in {'lc_username', 'lc_badges', 'lc_languages', 'lc_top_topics'}
}
lc_force_numeric = {
    k: v for k, v in leetcode_profile.items()
    if k in {
        'lc_total_solved', 'lc_easy_solved', 'lc_medium_solved', 'lc_hard_solved',
        'lc_contest_rating', 'lc_contest_rank', 'lc_top_percentage',
        'lc_contests_attended', 'lc_star_rating', 'lc_reputation', 'lc_ranking',
    }
}
lc_summary = save_platform_csvs(
    'leetcode', leetcode_profile,
    extra_numeric    = lc_force_numeric,
    extra_contextual = lc_force_contextual
)

# ── HACKERRANK ────────────────────────────────────────────────────────────────
# Star ratings per domain, badge count → numeric
# Raw skills text, username → contextual
hr_force_contextual = {
    k: v for k, v in hackerrank_profile.items()
    if k in {'hr_username', 'hr_skills_raw', 'hr_country'}
}
hr_force_numeric = {
    k: v for k, v in hackerrank_profile.items()
    if k.endswith('_stars') or k in {'hr_rank', 'hr_score', 'hr_total_badges'}
}
hr_summary = save_platform_csvs(
    'hackerrank', hackerrank_profile,
    extra_numeric    = hr_force_numeric,
    extra_contextual = hr_force_contextual
)

# ── STACK OVERFLOW ───────────────────────────────────────────────────────────
# Rep, counts, badge numbers, timestamps → numeric
# Display name, top tags string, profile URL → contextual
so_force_contextual = {
    k: v for k, v in stackoverflow_profile.items()
    if k in {'so_user_id', 'so_display_name', 'so_top_tags', 'so_profile_url'}
}
so_force_numeric = {
    k: v for k, v in stackoverflow_profile.items()
    if k in {
        'so_reputation', 'so_answer_count', 'so_question_count',
        'so_gold_badges', 'so_silver_badges', 'so_bronze_badges',
        'so_accepted_answers', 'so_avg_answer_score',
        'so_account_created', 'so_last_access',
    }
}
so_summary = save_platform_csvs(
    'stackoverflow', stackoverflow_profile,
    extra_numeric    = so_force_numeric,
    extra_contextual = so_force_contextual
)

# ── KAGGLE ────────────────────────────────────────────────────────────────────
# Medal counts, vote totals, ranking number → numeric
# Tier string, display name, dataset/notebook titles → contextual
kg_force_contextual = {
    k: v for k, v in kaggle_profile.items()
    if k in {'kg_username', 'kg_display_name', 'kg_tier',
              'kg_dataset_titles', 'kg_notebook_titles'}
}
kg_force_numeric = {
    k: v for k, v in kaggle_profile.items()
    if k in {
        'kg_ranking', 'kg_points', 'kg_gold_medals', 'kg_silver_medals',
        'kg_bronze_medals', 'kg_num_datasets', 'kg_dataset_votes',
        'kg_num_notebooks', 'kg_notebook_votes',
    }
}
kg_summary = save_platform_csvs(
    'kaggle', kaggle_profile,
    extra_numeric    = kg_force_numeric,
    extra_contextual = kg_force_contextual
)

# ── RESUME — contextual only ──────────────────────────────────────────────────
resume_ctx = {
    'email':       email,
    'resume_text': content['clean'],
    'resume_raw':  content['raw'],
}
pd.DataFrame([resume_ctx]).to_csv('resume_contextual.csv', index=False)
resume_summary = {'resume': {'numeric': 0, 'contextual': 3, 'status': 'saved',
                              'ctx_file': 'resume_contextual.csv'}}

# ── MASTER profile (all fields in one row — unchanged) ───────────────────────
all_data = {
    'email':          email,
    'linkedin_url':   linkedin,
    'github_url':     github,
    'leetcode_url':   leetcode,
    'hackerrank_url': hackerrank,
    'kaggle_url':     kaggle,
    'stackoverflow_url': stackoverflow,
    'resume_text':    content['clean'],
    **github_profile,
    **leetcode_profile,
    **hackerrank_profile,
    **linkedin_profile,
    **stackoverflow_profile,
    **kaggle_profile,
}
pd.DataFrame([all_data]).to_csv('candidate_profile.csv', index=False)

# ── GitHub repos (multi-row — unchanged) ─────────────────────────────────────
df_github_repos = pd.DataFrame(github_repos) if github_repos else pd.DataFrame()
df_github_repos.to_csv('github_repos.csv', index=False)

# ── Print summary ─────────────────────────────────────────────────────────────
all_summaries = {**li_summary, **gh_summary, **lc_summary,
                 **hr_summary, **so_summary, **kg_summary, **resume_summary}

print('✓ Per-platform transformation complete')
print()
print(f'  {'Platform':<16} {'Numeric fields':>16}  {'Contextual fields':>18}  Status')
print(f'  {"─" * 68}')
for platform, info in all_summaries.items():
    n = info.get('numeric', 0)
    c = info.get('contextual', 0)
    st = info.get('status', '')
    nf = info.get('num_file', '—')
    cf = info.get('ctx_file', '—')
    print(f'  {platform:<16} {n:>16}  {c:>18}  {st}')
    if st == 'saved':
        print(f'  {'':16}   → {nf}')
        print(f'  {'':16}   → {cf}')
print()
print(f'  candidate_profile.csv — {len(all_data)} columns (master, all platforms combined)')
print(f'  github_repos.csv      — {len(df_github_repos)} repos (multi-row)')

✓ Per-platform transformation complete

  Platform           Numeric fields   Contextual fields  Status
  ────────────────────────────────────────────────────────────────────
  linkedin                       15                  11  saved
                     → linkedin_numeric.csv
                     → linkedin_contextual.csv
  github                          7                   8  saved
                     → github_numeric.csv
                     → github_contextual.csv
  leetcode                       11                   4  saved
                     → leetcode_numeric.csv
                     → leetcode_contextual.csv
  hackerrank                     16                   3  saved
                     → hackerrank_numeric.csv
                     → hackerrank_contextual.csv
  stackoverflow                  10                   4  saved
                     → stackoverflow_numeric.csv
                     → stackoverflow_contextual.csv
  kaggle                          9          

In [ ]:
import json

# Extract the main profile data and remove the 'also_viewed' list for cleaner output
profile_data = raw.get('message', raw).copy()
if isinstance(profile_data, dict):
    profile_data.pop('also_viewed', None)

print(json.dumps(profile_data, indent=2))

{
  "associated_hashtag": [],
  "avatar_url": "https://media.licdn.com/dms/image/v2/D5603AQGvtbP33tidVg/profile-displayphoto-shrink_800_800/B56ZYcCfmAHQAk-/0/1744227139525?e=1777507200&v=beta&t=S6M8vYwJt6ZW25F5-z6Q6vJ45zA0uGwW1WkAWUgnujY",
  "background_picture": "",
  "birthday": {
    "day": "",
    "month": "",
    "year": ""
  },
  "certification": [
    {
      "authority": "LinkedIn",
      "company_id": "",
      "name": "Building an Adaptability Mindset in the Age of AI",
      "startedOn": {
        "day": null,
        "month": 2,
        "year": 2026
      },
      "url": "https://linkedin.com/company/1337"
    },
    {
      "authority": "LinkedIn",
      "company_id": "",
      "name": "Investing in Human Skills in the Age of AI",
      "startedOn": {
        "day": null,
        "month": 2,
        "year": 2026
      },
      "url": "https://linkedin.com/company/1337"
    }
  ],
  "company_id": "1035",
  "company_image": "https://media.licdn.com/dms/image/v2/D560BAQH32RJQ

In [ ]:
import pandas as pd
import json

# 1. Use the actual data from the LinkedIn response
# 'raw' was defined in the scraper cell
data = raw.get('message', raw)

# ---------------------------------------------------------
# Approach 1: The Main Profile DataFrame (Flattened)
# ---------------------------------------------------------
df_profile = pd.json_normalize(data)

# Define columns we want to see if they exist
core_cols = ['display_name', 'company_name', 'connections', 'country', 'current_company_name']
available_cols = [c for c in core_cols if c in df_profile.columns]

print("--- Main Profile DataFrame ---")
display(df_profile[available_cols].head())

# ---------------------------------------------------------
# Approach 2: Relational DataFrames for Nested Lists
# ---------------------------------------------------------

# Experience
exp_list = data.get('experience', [])
if exp_list:
    df_experience = pd.DataFrame(exp_list)
    print("\n--- Experience DataFrame ---")
    display(df_experience[['title', 'company_name', 'location', 'starts_at']].head() if 'title' in df_experience.columns else df_experience.head())

# Education
edu_list = data.get('education', [])
if edu_list:
    df_education = pd.DataFrame(edu_list)
    print("\n--- Education DataFrame ---")
    display(df_education[['university_name', 'fields_of_study']].head())

# Skills
skills_list = data.get('skills', [])
if skills_list:
    df_skills = pd.DataFrame({'skill_name': skills_list})
    print("\n--- Skills DataFrame ---")
    display(df_skills.head())


--- Main Profile DataFrame ---


,display_name,connections,country,current_company_name
0,Dewashish Dwivedi,5331,India,Microsoft



--- Experience DataFrame ---


,company_headcount_range,company_id,company_image,company_industry,company_name,company_url,company_website,employment_type,job_description,job_ended_on,job_location,job_location_city,job_location_country,job_location_county,job_location_state,job_started_on,job_still_working,job_title,raw_company_name,raw_job_title
0,10001-0,1035,https://media.licdn.com/dms/image/v2/D560BAQH3...,Software Development,Microsoft,https://www.linkedin.com/company/microsoft/,https://news.microsoft.com/,,[],,,Redmond,US,,Washington,3-2026,True,Threat Advisor,Microsoft,Threat Advisor
1,10001-0,1035,https://media.licdn.com/dms/image/v2/D560BAQH3...,Software Development,Microsoft,https://www.linkedin.com/company/microsoft/,https://news.microsoft.com/,Apprenticeship,[],,"Kanpur, Uttar Pradesh, India",Redmond,US,,Washington,6-2024,True,Beta Microsoft Student Ambassador,Microsoft,Beta Microsoft Student Ambassador
2,10001-0,1035,https://media.licdn.com/dms/image/v2/D560BAQH3...,Software Development,Microsoft,https://www.linkedin.com/company/microsoft/,https://news.microsoft.com/,Apprenticeship,[],3-2026,,Redmond,US,,Washington,2-2025,False,Azure Experience Insiders AXI(User Research),Microsoft,Azure Experience Insiders AXI(User Research)
3,10001-0,1035,https://media.licdn.com/dms/image/v2/D560BAQH3...,Software Development,Microsoft,https://www.linkedin.com/company/microsoft/,https://news.microsoft.com/,Apprenticeship,[],6-2024,"Kanpur, Uttar Pradesh, India",Redmond,US,,Washington,12-2023,False,Alpha Microsoft Student Ambassador,Microsoft,Alpha Microsoft Student Ambassador
4,10001-0,1035,https://media.licdn.com/dms/image/v2/D560BAQH3...,Software Development,Microsoft,https://www.linkedin.com/company/microsoft/,https://news.microsoft.com/,Apprenticeship,[],12-2023,,Redmond,US,,Washington,11-2023,False,Microsoft Student Ambassador,Microsoft,Microsoft Student Ambassador



--- Education DataFrame ---


,university_name,fields_of_study
0,PSIT Kanpur (Pranveer Singh Institute of Techn...,"[Bachelor of Technology - BTech, Artificial In..."
1,Ebenezer Higher Secondary School,"[12 (Intermidiate), PCM]"
2,Ebenezer H.R. Sec. School,"[10 (High School), Common subject]"
3,Google Academy,"[Certification , Mobile Sites]"
4,Manipal Global Education,"[Certification , Information Technology Projec..."



--- Skills DataFrame ---


,skill_name
0,Scikit-Learn
1,Machine Learning Algorithms
2,StreamLit
3,Web Applications
4,Front-End Design


In [ ]:
# Save the extracted LinkedIn DataFrames to CSV
if 'df_profile' in locals():
    df_profile.to_csv('linkedin_profile_full.csv', index=False)
    print("✓ Saved linkedin_profile_full.csv")

if 'df_experience' in locals():
    df_experience.to_csv('linkedin_experience.csv', index=False)
    print("✓ Saved linkedin_experience.csv")

if 'df_education' in locals():
    df_education.to_csv('linkedin_education.csv', index=False)
    print("✓ Saved linkedin_education.csv")

if 'df_skills' in locals():
    df_skills.to_csv('linkedin_skills.csv', index=False)
    print("✓ Saved linkedin_skills.csv")

✓ Saved linkedin_profile_full.csv
✓ Saved linkedin_experience.csv
✓ Saved linkedin_education.csv
✓ Saved linkedin_skills.csv


In [ ]:
import pandas as pd
from datetime import datetime

# Load the experience data
df_exp = pd.read_csv('linkedin_experience.csv')

def parse_li_date(date_str):
    if pd.isna(date_str) or str(date_str).strip() == "":
        return None
    try:
        # LinkedIn API in this notebook uses M-YYYY format
        return datetime.strptime(str(date_str), "%m-%Y")
    except:
        return None

def calculate_months(row):
    start = parse_li_date(row.get('job_started_on'))
    end = parse_li_date(row.get('job_ended_on'))
    is_current = row.get('job_still_working')

    if not start:
        return 0

    # If no end date and still working (or just no end date), use today
    if not end:
        if is_current or str(row.get('job_ended_on')).strip() == "":
            end = datetime.now()
        else:
            return 0

    diff = end - start
    return max(0, diff.days // 30)

# Apply calculation
df_exp['experience_months'] = df_exp.apply(calculate_months, axis=1)

# Save back to CSV
df_exp.to_csv('linkedin_experience.csv', index=False)

print("✓ Calculated experience durations")
display(df_exp[['job_title', 'job_started_on', 'job_ended_on', 'job_still_working', 'experience_months']].head())

✓ Calculated experience durations


/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,job_title,job_started_on,job_ended_on,job_still_working,experience_months
0,Threat Advisor,3-2026,NaN,True,1
1,Beta Microsoft Student Ambassador,6-2024,NaN,True,22
2,Azure Experience Insiders AXI(User Research),2-2025,3-2026,False,13
3,Alpha Microsoft Student Ambassador,12-2023,6-2024,False,6
4,Microsoft Student Ambassador,11-2023,12-2023,False,1


In [ ]:
import pandas as pd
import ast

def calculate_edu_months(row):
    try:
        # Parse dictionary-like strings into Python dicts
        start_raw = ast.literal_eval(row['started_on']) if isinstance(row['started_on'], str) else row['started_on']
        end_raw = ast.literal_eval(row['ended_on']) if isinstance(row['ended_on'], str) else row['ended_on']

        start_y = int(start_raw.get('year', 0))
        start_m = int(start_raw.get('month', 1))
        end_y = int(end_raw.get('year', 0))
        end_m = int(end_raw.get('month', 1))

        if start_y > 0 and end_y > 0:
            # Total months = (Years * 12) + Months
            total_months = (end_y * 12 + end_m) - (start_y * 12 + start_m)
            return max(0, total_months)
        return 0
    except Exception:
        return 0

# Load the education data
df_edu = pd.read_csv('linkedin_education.csv')

# Apply the calculation for months
df_edu['education_months'] = df_edu.apply(calculate_edu_months, axis=1)

# Save back to CSV
df_edu.to_csv('linkedin_education.csv', index=False)

print("✓ Calculated education duration in months")
display(df_edu[['university_name', 'started_on', 'ended_on', 'education_months']].head())

✓ Calculated education duration in months


,university_name,started_on,ended_on,education_months
0,PSIT Kanpur (Pranveer Singh Institute of Techn...,"{'month': 12, 'year': 2022}","{'month': 1, 'year': 2026}",37
1,Ebenezer Higher Secondary School,"{'month': 5, 'year': 2021}","{'month': 4, 'year': 2022}",11
2,Ebenezer H.R. Sec. School,"{'month': 4, 'year': 2019}","{'month': 3, 'year': 2020}",11
3,Google Academy,"{'month': 7, 'year': 2018}","{'month': 9, 'year': 2018}",2
4,Manipal Global Education,"{'month': 5, 'year': 2018}","{'month': 6, 'year': 2018}",1


In [ ]:
# ── Cell 13: Download all per-platform CSVs ───────────────────────────────────
from google.colab import files as colab_files
import os

# Organised by platform — numeric then contextual
PLATFORM_FILES = [
    # (filename,                    description)
    ('candidate_profile.csv',       'Master — all platforms combined'),
    ('github_repos.csv',            'GitHub repos — multi-row'),
    ('linkedin_numeric.csv',        'LinkedIn — basic numeric fields'),
    ('linkedin_contextual.csv',     'LinkedIn — basic contextual fields'),
    ('linkedin_profile_full.csv',   'LinkedIn — Full Profile (Flattened)'),
    ('linkedin_experience.csv',     'LinkedIn — Experience (with durations)'),
    ('linkedin_education.csv',      'LinkedIn — Education (with durations)'),
    ('linkedin_skills.csv',         'LinkedIn — Skills List'),
    ('github_numeric.csv',          'GitHub — numeric fields'),
    ('github_contextual.csv',       'GitHub — contextual fields'),
    ('leetcode_numeric.csv',        'LeetCode — numeric fields'),
    ('leetcode_contextual.csv',     'LeetCode — contextual fields'),
    ('hackerrank_numeric.csv',      'HackerRank — numeric fields'),
    ('hackerrank_contextual.csv',   'HackerRank — contextual fields'),
    ('stackoverflow_numeric.csv',   'Stack Overflow — numeric fields'),
    ('stackoverflow_contextual.csv','Stack Overflow — contextual fields'),
    ('kaggle_numeric.csv',          'Kaggle — numeric fields'),
    ('kaggle_contextual.csv',       'Kaggle — contextual fields'),
    ('resume_contextual.csv',       'Resume — text and email'),
]

print('Downloading per-platform CSVs...\n')
downloaded, skipped = [], []

for fname, desc in PLATFORM_FILES:
    if os.path.exists(fname):
        colab_files.download(fname)
        downloaded.append((fname, desc))
        print(f'  ↓ {fname:<40} {desc}')
    else:
        skipped.append((fname, desc))

if skipped:
    print()
    print('  Skipped (platform not available or not generated):')
    for fname, desc in skipped:
        print(f'  — {fname:<40} {desc}')

print()
print(f'✓ {len(downloaded)} files downloaded | {len(skipped)} skipped')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ↓ candidate_profile.csv                    Master — all platforms combined


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  ↓ github_contextual.csv                    GitHub — contextual fields


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ↓ resume_contextual.csv                    Resume — text and email

  Skipped (platform not available):
  — linkedin_numeric.csv                     LinkedIn — numeric fields
  — linkedin_contextual.csv                  LinkedIn — contextual fields
  — leetcode_numeric.csv                     LeetCode — numeric fields
  — leetcode_contextual.csv                  LeetCode — contextual fields
  — hackerrank_numeric.csv                   HackerRank — numeric fields
  — hackerrank_contextual.csv                HackerRank — contextual fields
  — stackoverflow_numeric.csv                Stack Overflow — numeric fields
  — stackoverflow_contextual.csv             Stack Overflow — contextual fields
  — kaggle_numeric.csv                       Kaggle — numeric fields
  — kaggle_contextual.csv                    Kaggle — contextual fields

✓ 5 files downloaded | 10 skipped
